# Tiền xử lý & Chuẩn hoá dữ liệu (format merge)

Mục tiêu của notebook này là **chuẩn hoá lại dữ liệu phòng trọ** về đúng format dùng cho các bài toán phân tích / hồi quy:

**Schema cuối cùng:**
- `url` *(string)*
- `title` *(string)*
- `price` *(number, VND/tháng)*
- `area` *(number, m²)*
- `address` *(string)*
- `description` *(string)*
- `source` *(string)*

Các đặc trưng khác như `posted_time`, `owner_name`, `phone` **sẽ bị loại bỏ** để tập trung vào bộ thuộc tính gọn cho mô hình.

Mỗi cell markdown giải thích **cách cell code bên dưới hoạt động** như một Data Scientist ghi chép lại pipeline.

## 1. Import thư viện & thiết lập chung

Cell dưới sẽ:
- Import các thư viện cần dùng: `os`, `re`, `numpy`, `pandas`.
- Thiết lập một vài tuỳ chọn hiển thị cho pandas để xem bảng cho dễ.
- Khai báo tên file CSV đầu vào (`INPUT_CSV`) và đầu ra (`OUTPUT_CSV`).

In [ ]:
import os
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# File CSV thô sinh từ crawler
INPUT_CSV = 'phongtro123.csv'  # đổi lại nếu file bạn tên khác

# File CSV sau khi chuẩn hoá theo format merge
OUTPUT_CSV = 'phongtro123_cleaned.csv'

## 2. Đọc dữ liệu từ file CSV gốc

Cell dưới sẽ:
- Kiểm tra xem `INPUT_CSV` có tồn tại hay không.
- Nếu có, đọc vào DataFrame `df` bằng `pd.read_csv`.
- In kích thước DataFrame và hiển thị vài dòng đầu để đối chiếu cấu trúc cột.

Lưu ý: Ở bước này **chưa chuẩn hoá gì cả**, chỉ là load dữ liệu gốc.

In [2]:
if not os.path.exists(INPUT_CSV):
    print(f'❌ File {INPUT_CSV} không tồn tại. Hãy kiểm tra lại đường dẫn / tên file.')
    df = pd.DataFrame()
else:
    df = pd.read_csv(INPUT_CSV)
    print('✅ Đã đọc dữ liệu, kích thước:', df.shape)
    display(df.head())

✅ Đã đọc dữ liệu, kích thước: (10305, 10)


,url,title,price,area,address,description,posted_time,owner_name,phone,source
0,https://phongtro123.com/chung-cu-go-thoai-ngoc...,"Chung cư Go Thoại Ngọc Hầu, TP: nhà sạch mới, ...",11 triệu/tháng,115 m2,"Đường Nguyễn Sơn, Phường Phú Thạnh, Quận Tân P...","Chung cư Go Thoại Ngọc Hầu, TP: nhà sạch mới, ...",10:36 23/11/2025,Phương Hằng,906500763,phongtro123
1,https://phongtro123.com/can-ho-dich-vu-xinh-iu...,Căn hộ dịch vụ xinh iu tiện nghi cho ai thích ...,5.3 triệu/tháng,30 m2,"Đường Đồng Đen, Quận Tân Bình, Hồ Chí Minh","Đồng Đen, Tân Bình Nội thất đầy đủ cho các nhu...",13:18 23/11/2025,Thanh Thanh,353988418,phongtro123
2,https://phongtro123.com/ky-tuc-xa-moi-xay-100-...,"KÝ TÚC XÁ MỚI XÂY 100% – NGUYỄN GIA TRÍ (D2), ...",1.5 triệu/tháng,35 m2,"168/44A Đường D2, Phường 25, Quận Bình Thạnh, ...","KÝ TÚC XÁ MỚI XÂY 100% – NGUYỄN GIA TRÍ (D2), ...",14:56 20/11/2025,Nguyễn Thị Thanh Thuỷ,913438412,phongtro123
3,https://phongtro123.com/chung-cu-prosper-phan-...,"Chung cư Prosper, Phan Văn Hớn, Q12: 74m2, 2p ...",9 triệu/tháng,74 m2,"Đường Phan Văn Hớn, Phường Tân Thới Nhất, Quận...","Chung cư Prosper, Phan Văn Hớn, Q12: 74m2, 2p ...",11:07 23/11/2025,Hoàng Oanh,905177098,phongtro123
4,https://phongtro123.com/tim-ban-cung-phong-stu...,TÌM BẠN CÙNG PHÒNG (STUDIO THẠNH MỸ LỢI),1 triệu/tháng,32 m2,"30 Đường số 2, Phường Thạnh Mỹ Lợi, Quận 2, Hồ...",Mình đang cần tìm GẤP một (1) bạn để share một...,12:39 23/11/2025,Alice Brooks,937020198,phongtro123


## 3. Chuẩn hoá cột `price` về số VND/tháng

Cell dưới định nghĩa hàm `parse_price_to_vnd` rồi áp dụng cho cột `price`:

**Cách hàm này chạy:**
1. Nhận vào một chuỗi giá, ví dụ: `'3.900.000 đ/tháng'`, `'4,5 triệu/tháng'`, `'2tr'`.
2. Đưa chuỗi về dạng đồng nhất (thường là chữ thường, thay `,` thành `.`).
3. Dùng regex để tách **phần số** và **đơn vị** (triệu / tr / nghìn / ngàn / k / đ / vnđ / vnd).
4. Tuỳ đơn vị:
   - `triệu` / `tr`  → nhân `1_000_000`.
   - `nghìn` / `ngàn` / `k` → nhân `1_000`.
   - `đ`, `vnđ`, `vnd` → giữ nguyên.
5. Nếu không tìm được đơn vị, cố gắng parse số thô (ví dụ trường hợp đã là `3900000`).
6. Hàm trả về **số VND/tháng** dạng `float`.

Sau đó ta:
- Tạo cột phụ `price_vnd` để nhìn thử.
- Ghi đè lại cột gốc `price` bằng giá trị numeric (VND/tháng).

In [3]:
def parse_price_to_vnd(price_text: str) -> float:
    '''
    Cách hàm này chạy như sau:
    - Nhận một chuỗi mô tả giá (ví dụ: '3.900.000 đ/tháng', '4,5 triệu/tháng', '2tr').
    - Chuẩn hoá chuỗi (lowercase, thay dấu phẩy thành chấm).
    - Dùng regex tách phần số & đơn vị (triệu / tr / nghìn / ngàn / k / đ / vnđ / vnd).
    - Chuyển về số VND bằng cách nhân với hệ số tương ứng.
    - Nếu không thấy đơn vị, cố gắng parse số thô.
    '''
    if not isinstance(price_text, str):
        return np.nan

    text = price_text.strip().lower()
    if not text:
        return np.nan

    # Thống nhất dấu thập phân
    text = text.replace(',', '.')

    # Regex bắt số + đơn vị
    pattern = re.compile(r'([0-9.]+)\s*(triệu|tr|nghìn|ngàn|k|đ|vnđ|vnd)')
    m = pattern.search(text)

    def _parse_number(num_str: str) -> float:
        """Hàm con: xử lý chuỗi số có nhiều dấu chấm.
        - Nếu có >= 2 dấu chấm → coi là dấu phân cách hàng nghìn, bỏ hết '.' rồi parse.
        - Ngược lại → parse float bình thường.
        """
        if num_str.count('.') >= 2:
            num_str_clean = num_str.replace('.', '')
            return float(num_str_clean)
        else:
            return float(num_str)

    if m:
        num_str, unit = m.groups()
        try:
            value = _parse_number(num_str)
        except ValueError:
            return np.nan

        unit = unit.strip()
        if unit in ['triệu', 'tr']:
            return value * 1_000_000
        elif unit in ['nghìn', 'ngàn', 'k']:
            return value * 1_000
        else:  # đ, vnđ, vnd
            return value

    # Nếu không bắt được đơn vị, thử lấy số thô
    num_match = re.search(r'[0-9.]+', text)
    if num_match:
        num_str = num_match.group(0)
        try:
            return _parse_number(num_str)
        except ValueError:
            return np.nan

    return np.nan


if not df.empty and 'price' in df.columns:
    df['price_vnd'] = df['price'].apply(parse_price_to_vnd)
    # Ghi đè cột gốc bằng giá numeric
    df['price'] = df['price_vnd']
    print('Ví dụ giá sau khi chuẩn hoá (VND/tháng):')
    display(df[['price_vnd', 'price']].head(10))
else:
    print('Không có cột price hoặc DataFrame trống.')

Ví dụ giá sau khi chuẩn hoá (VND/tháng):


,price_vnd,price
0,11000000.0,11000000.0
1,5300000.0,5300000.0
2,1500000.0,1500000.0
3,9000000.0,9000000.0
4,1000000.0,1000000.0
5,5800000.0,5800000.0
6,4000000.0,4000000.0
7,4500000.0,4500000.0
8,3800.0,3800.0
9,2900000.0,2900000.0


## 4. Chuẩn hoá cột `area` về số m²

Cell dưới định nghĩa hàm `parse_area_to_m2` rồi áp dụng cho cột `area`:

**Cách hàm này chạy:**
1. Nhận chuỗi như `'35 m2'`, `'40 m²'`, `'20.5m2'`.
2. Dùng regex để tìm **số đầu tiên** trong chuỗi.
3. Thay dấu phẩy bằng dấu chấm, xử lý trường hợp có nhiều dấu chấm.
4. Parse sang `float` biểu diễn diện tích m².

Sau đó ta tạo cột phụ `area_m2` rồi ghi đè cột gốc `area` bằng giá trị numeric này.

In [4]:
def parse_area_to_m2(area_text: str) -> float:
    '''
    Cách hàm này chạy như sau:
    - Nhận chuỗi mô tả diện tích (vd: '35 m2', '40 m²', '20.5m2').
    - Dùng regex lấy phần số đầu tiên.
    - Chuẩn hoá lại dấu thập phân (',' → '.').
    - Nếu có nhiều dấu chấm → gộp thành một số thập phân hợp lệ.
    - Trả về diện tích dạng float (đơn vị m²).
    '''
    if not isinstance(area_text, str):
        return np.nan
    text = area_text.strip().lower()
    if not text:
        return np.nan

    # Lấy cụm số đầu tiên trong chuỗi
    m = re.search(r'([0-9]+[0-9.,]*)', text)
    if not m:
        return np.nan

    num_str = m.group(1).replace(',', '.')
    try:
        parts = num_str.split('.')
        if len(parts) > 2:
            # Ví dụ: '1.234.56' → '1234.56'
            num_str_clean = ''.join(parts[:-1]) + '.' + parts[-1]
        else:
            num_str_clean = num_str
        return float(num_str_clean)
    except ValueError:
        return np.nan


if not df.empty and 'area' in df.columns:
    df['area_m2'] = df['area'].apply(parse_area_to_m2)
    # Ghi đè cột gốc bằng giá trị m²
    df['area'] = df['area_m2']
    print('Ví dụ diện tích sau khi chuẩn hoá (m²):')
    display(df[['area_m2', 'area']].head(10))
else:
    print('Không có cột area hoặc DataFrame trống.')

Ví dụ diện tích sau khi chuẩn hoá (m²):


,area_m2,area
0,115.0,115.0
1,30.0,30.0
2,35.0,35.0
3,74.0,74.0
4,32.0,32.0
5,32.0,32.0
6,35.0,35.0
7,25.0,25.0
8,20.0,20.0
9,20.0,20.0


## 5. Giữ đúng 7 cột theo format merge & loại bỏ cột thừa

Sau khi đã chuẩn hoá `price` và `area` thành dạng số, cell dưới sẽ:

1. **Loại bỏ các cột không cần**: `posted_time`, `owner_name`, `phone` (nếu tồn tại).
2. **Giữ đúng 7 cột** theo thứ tự:
   - `url`
   - `title`
   - `price`
   - `area`
   - `address`
   - `description`
   - `source`
3. Ép kiểu:
   - Các cột text: `url`, `title`, `address`, `description`, `source` → `string`/`object`.
   - Các cột số: `price`, `area` → kiểu số (`float`).

In [5]:
if not df.empty:
    # 1. Bỏ cột không dùng nếu có
    drop_cols = [c for c in ['posted_time', 'owner_name', 'phone'] if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)
        print('Đã xoá các cột không dùng:', drop_cols)

    # 2. Chọn đúng 7 cột theo schema
    target_cols = ['url', 'title', 'price', 'area', 'address', 'description', 'source']
    cols_exist = [c for c in target_cols if c in df.columns]
    missing = [c for c in target_cols if c not in df.columns]

    if missing:
        print('⚠️ Thiếu các cột sau trong dữ liệu gốc:', missing)

    df = df[cols_exist].copy()

    # 3. Ép kiểu cho đúng schema
    text_cols = [c for c in ['url', 'title', 'address', 'description', 'source'] if c in df.columns]
    for c in text_cols:
        df[c] = df[c].astype(str)

    for c in ['price', 'area']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    print('✅ Schema sau khi chuẩn hoá:')
    print(df.dtypes)
    display(df.head())
else:
    print('DataFrame trống - không thể chuẩn hoá schema.')

Đã xoá các cột không dùng: ['posted_time', 'owner_name', 'phone']
✅ Schema sau khi chuẩn hoá:
url             object
title           object
price          float64
area           float64
address         object
description     object
source          object
dtype: object


,url,title,price,area,address,description,source
0,https://phongtro123.com/chung-cu-go-thoai-ngoc...,"Chung cư Go Thoại Ngọc Hầu, TP: nhà sạch mới, ...",11000000.0,115.0,"Đường Nguyễn Sơn, Phường Phú Thạnh, Quận Tân P...","Chung cư Go Thoại Ngọc Hầu, TP: nhà sạch mới, ...",phongtro123
1,https://phongtro123.com/can-ho-dich-vu-xinh-iu...,Căn hộ dịch vụ xinh iu tiện nghi cho ai thích ...,5300000.0,30.0,"Đường Đồng Đen, Quận Tân Bình, Hồ Chí Minh","Đồng Đen, Tân Bình Nội thất đầy đủ cho các nhu...",phongtro123
2,https://phongtro123.com/ky-tuc-xa-moi-xay-100-...,"KÝ TÚC XÁ MỚI XÂY 100% – NGUYỄN GIA TRÍ (D2), ...",1500000.0,35.0,"168/44A Đường D2, Phường 25, Quận Bình Thạnh, ...","KÝ TÚC XÁ MỚI XÂY 100% – NGUYỄN GIA TRÍ (D2), ...",phongtro123
3,https://phongtro123.com/chung-cu-prosper-phan-...,"Chung cư Prosper, Phan Văn Hớn, Q12: 74m2, 2p ...",9000000.0,74.0,"Đường Phan Văn Hớn, Phường Tân Thới Nhất, Quận...","Chung cư Prosper, Phan Văn Hớn, Q12: 74m2, 2p ...",phongtro123
4,https://phongtro123.com/tim-ban-cung-phong-stu...,TÌM BẠN CÙNG PHÒNG (STUDIO THẠNH MỸ LỢI),1000000.0,32.0,"30 Đường số 2, Phường Thạnh Mỹ Lợi, Quận 2, Hồ...",Mình đang cần tìm GẤP một (1) bạn để share một...,phongtro123


## 6. Lưu file CSV đã chuẩn hoá theo format merge

Cell dưới sẽ:
- Ghi DataFrame hiện tại (`df`) ra file `OUTPUT_CSV`.
- Đảm bảo file chỉ còn **7 cột** đúng thứ tự: 
  `url, title, price, area, address, description, source`.

Đây sẽ là file dùng cho các bước **EDA nâng cao, feature engineering, regular regression, v.v.**

In [6]:
if not df.empty:
    df.to_csv(OUTPUT_CSV, index=False)
    print(f'💾 Đã lưu dữ liệu đã chuẩn hoá vào: {OUTPUT_CSV}')
    print('Số dòng, số cột:', df.shape)
else:
    print('DataFrame trống - không có gì để lưu.')

💾 Đã lưu dữ liệu đã chuẩn hoá vào: phongtro123_merged.csv
Số dòng, số cột: (10305, 7)
